# Bonus 03: Extract a regional mesh-cube with Iris

While GeoVista provides the efficient tools for mesh region extraction, it and Iris know nothing about one another. So, to calculate a regionally-extracted _Iris cube_, GeoVista can do the hard work of determining the subset of cells required, but you must then "reconstruct" an Iris cube from that information. For now, at least, there are no ready-made tools for this (either in Iris or GeoVista).  

The process requires a few steps, which we can summarise as :
  1. Record, on the original global PolyData, the original face indices of each of the cells
  1. Perform extraction (by BBox or otherwise) to get a regional PolyData
  1. Get the face-indices of the selected cells from the regional PolyData  
  1. Index the Iris cube with the selected indices, on the mesh dimension, to extract the regional parts
  1. Construct and attach a suitable Iris mesh to represent the extracted region

(Note: the last step itself is not strictly necessary. It may be sufficent to have a regional data cube with a notional "mesh dimension", but which does not possess an actual Iris mesh.)

---
**First, we just repeat some of the imports and code from 'Sec_05_Region_Extraction'**

to get a global mesh-cube and a PolyData derived from it.

In [1]:
# Notebook warning controls
import warnings

# Suppress repeated NumPy masked-array casting warnings from sample data loading.
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in cast",
    category=RuntimeWarning,
)
# Suppress Iris legacy date-precision future warning in tutorial output.
warnings.filterwarnings(
    "ignore",
    message="You are using legacy date precision for Iris units*",
    category=FutureWarning,
)

# Use microsecond date precision to align with upcoming Iris default behaviour.
try:
    import iris
    iris.FUTURE.date_microseconds = True
except Exception:
    pass

# Fetch a single cube of test data
from support.testdata_fetching import lfric_rh_singletime_2d
lfric_rh = lfric_rh_singletime_2d()
#lfric_rh

In [2]:
# Convert to a PyVista.PolyData
from support.pv_conversions import pv_from_lfric_cube
pv_global_rh = pv_from_lfric_cube(lfric_rh)
#pv_global_rh

In [3]:
# Define a lat-lon region to extract
from geovista.geodesic import BBox

lon_min = -15
lon_max = 55
lat_min = -5
lat_max = 35

bbox = BBox(lons=[lon_min, lon_max, lon_max, lon_min], lats=[lat_min, lat_min, lat_max, lat_max])

In [4]:
# 'Apply' the regiona to the PolyData object
pv_regional_rh = bbox.enclosed(pv_global_rh)
pv_regional_rh

PolyData (0x314253e80)
  N Cells:    953
  N Points:   1019
  N Strips:   0
  X Bounds:   4.743e-01, 1.000e+00
  Y Bounds:   -2.588e-01, 8.128e-01
  Z Bounds:   -1.207e-01, 6.635e-01
  N Arrays:   6

---
## Now the meshcube extraction, as a sequence of steps ...


### Step 1: Add an auxiliary array to our _global_ PolyData
This is to record, for each PolyData cell, the original (face) index which that cell has.

Note: We use numpy.arange() to construct a counting sequence, and make this a new array on the PolyData object, by assigning to a index-name.

In [5]:
import numpy as np
face_inds = np.arange(pv_global_rh.n_cells)
# Assign PolyData[<a-string>] to create a new attached array.
pv_global_rh.cell_data['original_face_indices'] = face_inds

---

### Step 2: Extract with your Bbox to get a regional PolyData, and show the result.
"Enclose" your global data with the Bounding Box as done in  [Section 05](./05_RegionExtraction.ipynb) and for demonstration above.

In [6]:
# 'Apply' the region to the PolyData object.
pv_regional_rh = bbox.enclosed(pv_global_rh)
pv_regional_rh

PolyData (0x314253fa0)
  N Cells:    953
  N Points:   1019
  N Strips:   0
  X Bounds:   4.743e-01, 1.000e+00
  Y Bounds:   -2.588e-01, 8.128e-01
  Z Bounds:   -1.207e-01, 6.635e-01
  N Arrays:   7

You can see that the new version of the extracted (regional) data now has an ***extra*** attached data array, "`original_face_indices`", which is derived from the one we added to the global data, and which holds the face indices of the _selected_ cells.

---

### Step 3: Fetch the indices array from the regional PolyData, by indexing with the array name.
and show the result.

In [7]:
# Get the remaining face indices, to use for indexing the Cube.
region_indices = pv_regional_rh["original_face_indices"]
region_indices

pyvista_ndarray([   40,    41,    42,    43,    44,    45,    46,    47,
                    82,    83,    84,    85,    86,    87,    88,    89,
                    90,    91,    92,    93,    94,    95,   125,   126,
                   127,   128,   129,   130,   131,   132,   133,   134,
                   135,   136,   137,   138,   139,   140,   141,   142,
                   143,   167,   168,   169,   170,   171,   172,   173,
                   174,   175,   176,   177,   178,   179,   180,   181,
                   182,   183,   184,   185,   186,   187,   188,   189,
                   190,   191,   210,   211,   212,   213,   214,   215,
                   216,   217,   218,   219,   220,   221,   222,   223,
                   224,   225,   226,   227,   228,   229,   230,   231,
                   232,   233,   234,   235,   236,   237,   238,   239,
                   256,   257,   258,   259,   260,   261,   262,   263,
                   264,   265,   266,   267,   268,

This contains the original face-indices of all the cells which fall within the region, _i.e. which faces those were in the global mesh_.

We can now apply these via indexing, to select only those cells *from the Iris cube*.

### Step 4: Apply these cells as an index to the 'mesh dimension' of the original Iris lfric-rh cube
.. and print that out.

In [8]:
lfric_rh_region = lfric_rh[..., region_indices]
lfric_rh_region

Relative Humidity At Screen Level (1),--
Shape,953
Auxiliary coordinates,
latitude,x
longitude,x
Scalar coordinates,
forecast_period,21600 seconds
forecast_reference_time,2021-03-24 00:00:00
time,2021-03-24 06:00:00
Cell methods,
0,time: point


This new cube contains the mesh data within our selected region.

However, there is a catch here: Once indexed, our cube ***no longer has a mesh***. You can see this in the printout, which lists "Auxiliary coordinates" but no "Mesh coordinates". This problem will probably be fixed in future.  See [here in the Iris docs](https://scitools-iris.readthedocs.io/en/latest/further_topics/ugrid/operations.html#region-extraction) for a discussion. For now, what we need to do is to re-create a mesh for the regional cube.
We do that in a few further steps ...

---

### Step 5a: Get the X and Y-axis coordinates from the region cube.

In [9]:
x_coord = lfric_rh_region.coord('longitude')
y_coord = lfric_rh_region.coord('latitude')

### Step 5b: Create a new `iris.mesh.MeshXY` object from the X/Y coordinates


In [10]:
from iris.mesh import MeshXY
mesh = MeshXY.from_coords(x_coord, y_coord)

#print(mesh)


---
### Step 5c:  Call `Mesh.to_MeshCoords` to create a pair of `MeshCoord`s containing this mesh
Note: You must specify the keyword `location="face"`:  This matches the data location of the original data, i.e. the data cells are faces.

In [11]:
mesh_coords = mesh.to_MeshCoords(location="face")
mesh_coords

(<MeshCoord: longitude / (degrees)  mesh(<MeshXY object at 0x3142aab10>) location(face)  <lazy>+bounds  shape(953,)>,
 <MeshCoord: latitude / (degrees)  mesh(<MeshXY object at 0x3142aab10>) location(face)  <lazy>+bounds  shape(953,)>)

---
### Step 5d : Use `Cube.remove_coord` and `Cube.add_aux_coord` to replace each AuxCoord with its corresponding `MeshCoord` from the previous step. 
Note: For 'add_aux_coord', you also need to specify the relevant cube dimension(s). See Iris documentation for [Cube.add_aux_coord](https://scitools-iris.readthedocs.io/en/latest/generated/api/iris/cube.html?highlight=add_aux_coord#iris.cube.Cube.add_aux_coord)  
.. and show the cube ..

In [12]:
lfric_rh_region.remove_coord('longitude')
lfric_rh_region.remove_coord('latitude')

In [13]:
xco, yco = mesh_coords

lfric_rh_region.add_aux_coord(xco, 0)
lfric_rh_region.add_aux_coord(yco, 0)

# Result : a regional Mesh-Cube with a subset of the original faces.
lfric_rh_region

Relative Humidity At Screen Level (1),--
Shape,953
Mesh coordinates,
latitude,x
longitude,x
Mesh,
name,unknown
location,face
Scalar coordinates,
forecast_period,21600 seconds
forecast_reference_time,2021-03-24 00:00:00


---

## Plot this to see what we got.
Use the techniques as above, converting with `pv_from_lfric_cube` and plotting.


In [14]:
pv = pv_from_lfric_cube(lfric_rh_region)
pv.plot()

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/lb788/miniconda3/envs/lfric-mesh/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x107346c00> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/lb788/miniconda3/envs/lfric-mesh/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x107346c00> is already entered


Widget(value='<iframe src="http://localhost:50721/index.html?ui=P_0x314277470_0&reconnect=auto" class="pyvista…

----

**Investigation:** It is useful to add some extra background information to make this more visible.

As a minimum you can use `plotter.add_coastlines()`.

Another useful one is `plotter.add_base_layer()`  
**Question :  what does that actually do ?**

In [15]:
from geovista import GeoPlotter
plotter = GeoPlotter()
plotter.add_mesh(pv)
plotter.add_coastlines()
plotter.add_base_layer()
plotter.show()

Widget(value='<iframe src="http://localhost:50721/index.html?ui=P_0x316cd9f70_1&reconnect=auto" class="pyvista…